In [ ]:
#housekeeping

from google.colab import drive
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.models import load_model
from keras import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.preprocessing import image
from keras.layers import BatchNormalization

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from IPython.display import clear_output

In [ ]:
#mount drive and initialize global paths

#mount drive and save dataset directory
drive.mount('/content/drive')
dataset = '/content/drive/My Drive/PVdatasetUpdated'

#global paths
drive_path = '/content/drive/My Drive'
symptoms_json_path = '/content/drive/My Drive/plantVis3/disease_symptoms.json'
plant_village_path = '/content/drive/My Drive/PVdatasetUpdated'
dataset_path = '/content/drive/My Drive/plantVis4/data.csv'
model_path = '/content/drive/My Drive/plantVis4/model.keras'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#convert file of files of images into a single, usable dataframe of labeled data
#commented out cuz it's already made the csv and not needed anymore

#36 unique symptoms, including 'Healthy'

#'''
#dataframe that associates disease names to symptoms
disease_to_symptoms_df = pd.read_json(symptoms_json_path)

#func to get a list of symptoms associated with a disease name
def get_symptoms(df, disease):
    if disease == "Healthy":
      return ['Healthy']
    row = df.loc[df["disease"] == disease, "symptoms"]
    return row.iloc[0] if not row.empty else None

#gather all images and their labels into one dataframe
image_paths = []
labels = []
symptoms = []
unique_labels = []
unique_symptoms = []
for plant_type in os.listdir(plant_village_path):
  plant_path = os.path.join(plant_village_path, plant_type)
  if not os.path.isdir(plant_path):
    continue

  for partition in os.listdir(plant_path):
    partition_path = os.path.join(plant_path, partition)
    if not os.path.isdir(partition_path):
      continue

    for label in os.listdir(partition_path):
      label_path = os.path.join(partition_path, label)
        #get the names of folders since they are labels: Healthy, Rust, etc.
      if not os.path.isdir(label_path):
        continue

      symptom_list = get_symptoms(disease_to_symptoms_df, label)
      if symptom_list == None:
        continue

      for img_file in os.listdir(label_path):
        img_file_path = os.path.join(label_path, img_file)
        if os.path.isfile(img_file_path) and img_file.lower().endswith(('jpg', 'jpeg', '.png')):
          #add full path to img_file to image_paths list
            #and label of the image (taken from folder name) to labels list
          image_paths.append(img_file_path)
          if label not in labels:
            unique_labels.append(label)
          labels.append(label)

          for symptom in symptom_list:
            if symptom not in unique_symptoms:
              unique_symptoms.append(symptom)
          symptoms.append(symptom_list)

symptoms_multihot = []
for symptom_list in symptoms:
  symptom_index_list=[0] * len(unique_symptoms) #initialize to list of 36 0s
  for symptom in symptom_list:
    #if the symptom is in the list, change corresponding value in symptom_index_list to 1 for multihot
    symptom_index = unique_symptoms.index(symptom)
    symptom_index_list[symptom_index] = 1
  symptoms_multihot.append(symptom_index_list)
symptoms_df = pd.DataFrame(symptoms_multihot, columns=unique_symptoms)

#make new labels based on
df = pd.DataFrame({
    "image": image_paths,
    "disease": labels
})
df = pd.concat([df, symptoms_df], axis=1)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv(dataset_path, index=False)

#test that everything associated properly and can be easily accessed/syntax reference
datfram = pd.read_csv(dataset_path)
samp = datfram.sample(n=1).iloc[0]
cols_with_1 = samp[samp == 1].index.tolist()
print(samp.image,"\n",samp.disease,"\n", cols_with_1)
#passes the vibe check, symptoms were properly associated with images and labels
#'''

/content/drive/My Drive/PVdatasetUpdated/Corn (Maize)/Train/Healthy/1d81ce56-55bc-4037-8d1f-6d1ea0f94698___R.S_HL 5530 copy 2_flipLR.jpg 
 Healthy 
 ['Healthy']


In [ ]:
#make training data lists from training data csv

dataset = pd.read_csv(dataset_path)
#print(dataset.columns[2:])
#dataset.columns[2:] gives the names of only the symptom columns

image_paths = dataset["image"].values
labels = dataset[dataset.columns[2:]]
label_names = list(dataset.columns[2:])

ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

def load_and_preprocess(path, label):
  img = tf.io.read_file(path)
  img = tf.image.decode_jpeg(img, channels=3)
  img = tf.image.resize(img, (256, 256))
  img = img / 255.0
  return img, label

ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
#ds = ds.shuffle(1000, seed=42).batch(32).prefetch(tf.data.AUTOTUNE)

# Split into train/val/test
  #86%, 7%, 7%
dataset_size = len(dataset)

train_size = int(0.84 * dataset_size)
val_size   = int(0.07 * dataset_size)
test_size  = dataset_size - train_size - val_size   # avoids rounding errors

ds = ds.shuffle(1000, seed=42)

train = ds.take(train_size)
val   = ds.skip(train_size).take(val_size)
test  = ds.skip(train_size + val_size)

train_ds = train.batch(32).prefetch(tf.data.AUTOTUNE)
val_ds   = val.batch(32).prefetch(tf.data.AUTOTUNE)
test_ds  = test.batch(32).prefetch(tf.data.AUTOTUNE)


print(dataset_size)
print(train_size)
print(val_size)
print(len(dataset) - (train_size + val_size))
print(train_size + val_size + (len(dataset) - (train_size + val_size)))




single = train_ds.unbatch().take(1)
for img, label in single:
    print(img.shape, label)
print(label_names)


50952
42799
3566
4587
50952
(256, 256, 3) tf.Tensor([0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0], shape=(36,), dtype=int64)
['Tan', 'Sunken', 'Black-Lesions', 'Black-Veins', 'Rotting', 'Brown', 'Wilt', 'Blotchy', 'Healthy', 'Oily', 'Yellow', 'Greasy', 'Black', 'Scorched', 'Purple', 'Brown-Lesions', 'Watery', 'Halo', 'Spots', 'Distortion', 'Sunscald', 'Powdery', 'Fuzzy', 'Elliptical-Lesions', 'Parallel-Lesions', 'Green-Lesions', 'Spores', 'Yellow-Lesions', 'Discolored-Streaks', 'White-Mold', 'Circular', 'Rings', 'Collapsed', 'Shriveled', 'Black-Specks', 'Discolored']


In [ ]:
#build model arhcitecture
'''
model = Sequential()

model = Sequential()

model.add(Conv2D(filters=16, kernel_size=(5, 5), activation="relu", input_shape=(255,255,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.2))

model.add(Conv2D(filters=32, kernel_size=(5, 5), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Conv2D(filters=64, kernel_size=(5, 5), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Conv2D(filters=64, kernel_size=(5, 5), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(36, activation='sigmoid'))

model.summary()
#'''

'\nmodel = Sequential()\n\nmodel = Sequential()\n\nmodel.add(Conv2D(filters=16, kernel_size=(5, 5), activation="relu", input_shape=(255,255,3)))\nmodel.add(BatchNormalization())\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\nmodel.add(Dropout(0.2))\n\nmodel.add(Conv2D(filters=32, kernel_size=(5, 5), activation=\'relu\'))\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\nmodel.add(BatchNormalization())\nmodel.add(Dropout(0.2))\n\nmodel.add(Conv2D(filters=64, kernel_size=(5, 5), activation="relu"))\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\nmodel.add(BatchNormalization())\nmodel.add(Dropout(0.2))\n\nmodel.add(Conv2D(filters=64, kernel_size=(5, 5), activation=\'relu\'))\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\nmodel.add(BatchNormalization())\nmodel.add(Dropout(0.2))\n\nmodel.add(Flatten())\nmodel.add(Dense(128, activation=\'relu\'))\nmodel.add(Dropout(0.5))\nmodel.add(Dense(64, activation=\'relu\'))\nmodel.add(Dropout(0.5))\nmodel.add(Dense(36, activation=\'sigmoid\'))\n\nmodel.summary()\n#'

In [ ]:
'''
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'recall'])
model.save(model_path)
'''

"\nmodel.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'recall'])\nmodel.save(model_path)\n"

In [ ]:
model = load_model(model_path)

In [ ]:
class SavePeriodic(tf.keras.callbacks.Callback):
    def __init__(self, steps_interval=200):
        super().__init__()
        self.steps_interval = steps_interval
        self.batch_losses = []
        self.batch_accs = []

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}

        # Store per-batch metrics
        if "loss" in logs:
            self.batch_losses.append(logs["loss"])
        if "accuracy" in logs:
            self.batch_accs.append(logs["accuracy"])

        # Autosave + plot periodically
        if batch % self.steps_interval == 0 and batch > 0:
            self.model.save(model_path)
            print(f"Autosave at batch {batch}")

            # Plot updated graphs
            self.plot_metrics()

    def plot_metrics(self):
        plt.figure(figsize=(10, 4))

        # Loss curve
        plt.subplot(1, 2, 1)
        plt.plot(self.batch_losses)
        plt.title("Batch Loss History")
        plt.xlabel("Batch")
        plt.ylabel("Loss")

        # Accuracy curve (only if available)
        plt.subplot(1, 2, 2)
        if self.batch_accs:
            plt.plot(self.batch_accs)
            plt.title("Batch Accuracy History")
            plt.xlabel("Batch")
            plt.ylabel("Accuracy")

        plt.tight_layout()
        plt.show()

In [ ]:
'''
history = model.fit(train_ds, epochs=5, validation_data=val_ds, batch_size=64, callbacks=[SavePeriodic(64)])

#plot the training and validation accuracy and loss at each epoch
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'y', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

acc = history.history['acc']
val_acc = history.history['val_acc']
plt.plot(epochs, acc, 'y', label='Training acc')
plt.plot(epochs, val_acc, 'r', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
#'''

" \nhistory = model.fit(train_ds, epochs=5, validation_data=val_ds, batch_size=64, callbacks=[SavePeriodic(64)])\n\n#plot the training and validation accuracy and loss at each epoch\nloss = history.history['loss']\nval_loss = history.history['val_loss']\nepochs = range(1, len(loss) + 1)\nplt.plot(epochs, loss, 'y', label='Training loss')\nplt.plot(epochs, val_loss, 'r', label='Validation loss')\nplt.title('Training and validation loss')\nplt.xlabel('Epochs')\nplt.ylabel('Loss')\nplt.legend()\nplt.show()\n\nacc = history.history['acc']\nval_acc = history.history['val_acc']\nplt.plot(epochs, acc, 'y', label='Training acc')\nplt.plot(epochs, val_acc, 'r', label='Validation acc')\nplt.title('Training and validation accuracy')\nplt.xlabel('Epochs')\nplt.ylabel('Accuracy')\nplt.legend()\nplt.show()\n#"

In [ ]:
from pandas._libs.tslibs.offsets import Micro

numC = 35

metrics = [
    tf.keras.metrics.BinaryAccuracy(),

    tf.keras.metrics.Precision(),     # per-label (macro by default internally)
    tf.keras.metrics.Recall(),
    tf.keras.metrics.F1Score(threshold=0.5),

    tf.keras.metrics.AUC(curve="PR", multi_label=True),
    tf.keras.metrics.AUC(curve="ROC", multi_label=True)
]




model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=metrics
)

In [ ]:
import os

eval_path = '/content/drive/My Drive/model4evaluation.csv'

class PrintBatchMetrics(tf.keras.callbacks.Callback):
    def on_test_batch_end(self, batch, logs=None):
        print(f"Batch {batch}: {logs}")

temp_test = test_ds.take(64)
history = model.evaluate(
    temp_test,#test_ds
    return_dict=True,
    verbose=1
)

# Ensure directory exists
os.makedirs(os.path.dirname(eval_path), exist_ok=True)

# history is a dict: {"loss": ..., "precision": ..., ... }
with open(eval_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    for key, value in history.items():
        writer.writerow([key, value])

print(f"\nEvaluation results saved to: {eval_path}")

ValueError: math domain error